In [ ]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq
)
from datasets import Dataset
import evaluate
import numpy as np
import subprocess
import tempfile
import json
import os
from tqdm import tqdm

CHECKPOINT_PATH = "./nllb-specialization-5e4-do-0.3-5epochs/checkpoint-9380"

# Test data files
TEST_NEP_FILE = "dataset/data-nmt/ne-test.tsv"
TEST_HIN_FILE = "dataset/data-nmt/tmg-test.tsv"

# Language codes
NEP_LANG = "npi_Deva"
HIN_LANG = "hin_Deva"

# Generation parameters
BATCH_SIZE = 4
GENERATION_MAX_LENGTH = 128
GENERATION_NUM_BEAMS = 4
COMET_MODEL = "Unbabel/wmt22-comet-da"
COMET_BATCH_SIZE = 4

def load_data_files(nep_file, hin_file):
    with open(nep_file, 'r', encoding='utf-8') as f:
        nep_texts = [line.strip() for line in f.readlines()]
    with open(hin_file, 'r', encoding='utf-8') as f:
        hin_texts = [line.strip() for line in f.readlines()]
    return nep_texts, hin_texts

def create_test_datasets(test_nep, test_hin):
    test_nep_data, test_hin_data = load_data_files(test_nep, test_hin)
    
    test_nep2hin = Dataset.from_dict({
        "source": test_nep_data,
        "target": test_hin_data,
        "src_lang": [NEP_LANG] * len(test_nep_data),
        "tgt_lang": [HIN_LANG] * len(test_nep_data)
    })
    
    test_hin2nep = Dataset.from_dict({
        "source": test_hin_data,
        "target": test_nep_data,
        "src_lang": [HIN_LANG] * len(test_hin_data),
        "tgt_lang": [NEP_LANG] * len(test_hin_data)
    })
    
    return test_nep2hin, test_hin2nep

def create_preprocess_function(tokenizer):
    def preprocess_data(examples):
        sources = examples["source"]
        targets = examples["target"]
        src_langs = examples["src_lang"]
        tgt_langs = examples["tgt_lang"]
        
        input_ids = []
        attention_masks = []
        labels = []
        
        for source, target, src_lang, tgt_lang in zip(sources, targets, src_langs, tgt_langs):
            tokenizer.src_lang = src_lang
            source_encoding = tokenizer(source, max_length=128, truncation=True, padding=False)
            
            target_tokens = tokenizer(
                target, max_length=126, truncation=True, 
                padding=False, add_special_tokens=False
            )["input_ids"]
            
            tgt_lang_id = tokenizer.convert_tokens_to_ids(tgt_lang)
            target_ids = [tgt_lang_id] + target_tokens + [tokenizer.eos_token_id]
            
            input_ids.append(source_encoding["input_ids"])
            attention_masks.append(source_encoding["attention_mask"])
            labels.append(target_ids)
        
        return {
            "input_ids": input_ids,
            "attention_mask": attention_masks,
            "labels": labels
        }
    return preprocess_data

def compute_comet_score(sources, predictions, references):
    try:
        with tempfile.NamedTemporaryFile(mode='w', encoding='utf-8', suffix='.txt', delete=False) as src_f, \
             tempfile.NamedTemporaryFile(mode='w', encoding='utf-8', suffix='.txt', delete=False) as hyp_f, \
             tempfile.NamedTemporaryFile(mode='w', encoding='utf-8', suffix='.txt', delete=False) as ref_f:
            
            src_f.write('\n'.join(sources))
            hyp_f.write('\n'.join(predictions))
            ref_f.write('\n'.join(references))
            
            src_file = src_f.name
            hyp_file = hyp_f.name
            ref_file = ref_f.name
        
        cmd = [
            'comet-score',
            '-s', src_file,
            '-t', hyp_file,
            '-r', ref_file,
            '--model', COMET_MODEL,
            '--batch_size', str(COMET_BATCH_SIZE),
            '--quiet',
            '--only_system'
        ]
        
        result = subprocess.run(cmd, capture_output=True, text=True, check=True)
        output = result.stdout.strip()
        
        try:
            score_data = json.loads(output)
            score = float(score_data.get('system_score', output))
        except:
            score = float(output)
        
        os.unlink(src_file)
        os.unlink(hyp_file)
        os.unlink(ref_file)
        
        return score
        
    except Exception as e:
        print(f"Warning: COMET computation failed: {str(e)}")
        return 0.0

def compute_metrics(tokenizer, eval_preds):
    predictions, labels = eval_preds
    
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    
    decoded_preds = [pred.strip() for pred in decoded_preds]
    decoded_labels = [label.strip() for label in decoded_labels]
    
    sacrebleu = evaluate.load("sacrebleu")
    chrf = evaluate.load("chrf")
    meteor = evaluate.load("meteor")
    
    sacrebleu_result = sacrebleu.compute(predictions=decoded_preds, references=decoded_labels)
    chrf_result = chrf.compute(predictions=decoded_preds, references=decoded_labels, word_order=0)
    chrfpp_result = chrf.compute(predictions=decoded_preds, references=decoded_labels, word_order=2)
    meteor_result = meteor.compute(predictions=decoded_preds, references=decoded_labels)
    
    return {
        "sacrebleu": sacrebleu_result["score"],
        "chrf": chrf_result["score"],
        "chrfpp": chrfpp_result["score"],
        "meteor": meteor_result["meteor"],
    }

def evaluate_direction(trainer, test_dataset, direction_name):
    print(f"\nEvaluating {direction_name}...")
    
    # Get original data
    original_sources = test_dataset["source"]
    original_targets = test_dataset["target"]
    
    # Preprocess
    preprocess_fn = create_preprocess_function(trainer.tokenizer)
    processed_dataset = test_dataset.map(
        preprocess_fn, batched=True, 
        remove_columns=test_dataset.column_names
    )
    
    # Get basic metrics
    metrics = trainer.evaluate(eval_dataset=processed_dataset)
    
    # Get predictions for COMET
    predictions = trainer.predict(test_dataset=processed_dataset)
    decoded_preds = trainer.tokenizer.batch_decode(predictions.predictions, skip_special_tokens=True)
    decoded_preds = [pred.strip() for pred in decoded_preds]
    
    # Compute COMET
    comet_score = compute_comet_score(original_sources, decoded_preds, original_targets)
    
    return {
        'sacrebleu': metrics['eval_sacrebleu'],
        'chrf': metrics['eval_chrf'],
        'chrfpp': metrics['eval_chrfpp'],
        'meteor': metrics['eval_meteor'],
        'comet': comet_score
    }

def main():
    print("\n" + "="*80)
    print("CHECKPOINT EVALUATION")
    print("="*80)
    print(f"Checkpoint: {CHECKPOINT_PATH}")
    print("="*80 + "\n")
    
    # Load model and tokenizer
    print("Loading model and tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT_PATH)
    model = AutoModelForSeq2SeqLM.from_pretrained(CHECKPOINT_PATH)
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    print(f"Model loaded on: {device}\n")
    
    # Create test datasets
    print("Creating test datasets...\n")
    test_nep2hin, test_hin2nep = create_test_datasets(TEST_NEP_FILE, TEST_HIN_FILE)
    
    # Setup trainer
    training_args = Seq2SeqTrainingArguments(
        output_dir="./temp_eval",
        per_device_eval_batch_size=BATCH_SIZE,
        predict_with_generate=True,
        generation_max_length=GENERATION_MAX_LENGTH,
        generation_num_beams=GENERATION_NUM_BEAMS,
        fp16=torch.cuda.is_available(),
        disable_tqdm=False,  # Enable progress bar
    )
    
    data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model, padding=True)
    
    trainer = Seq2SeqTrainer(
        model=model,
        args=training_args,
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=lambda x: compute_metrics(tokenizer, x),
    )
    
    # Evaluate both directions
    nep2hin_results = evaluate_direction(trainer, test_nep2hin, "NEPALI → HINDI")
    hin2nep_results = evaluate_direction(trainer, test_hin2nep, "HINDI → NEPALI")
    
    # Print results
    print("\n" + "="*80)
    print("NEPALI → HINDI RESULTS")
    print("="*80)
    print(f"SacreBLEU: {nep2hin_results['sacrebleu']:.4f}")
    print(f"ChrF:      {nep2hin_results['chrf']:.4f}")
    print(f"ChrF++:    {nep2hin_results['chrfpp']:.4f}")
    print(f"METEOR:    {nep2hin_results['meteor']:.4f}")
    print(f"COMET:     {nep2hin_results['comet']:.4f}")
    
    print("\n" + "="*80)
    print("HINDI → NEPALI RESULTS")
    print("="*80)
    print(f"SacreBLEU: {hin2nep_results['sacrebleu']:.4f}")
    print(f"ChrF:      {hin2nep_results['chrf']:.4f}")
    print(f"ChrF++:    {hin2nep_results['chrfpp']:.4f}")
    print(f"METEOR:    {hin2nep_results['meteor']:.4f}")
    print(f"COMET:     {hin2nep_results['comet']:.4f}")
    
    print("\n" + "="*80)
    print("AVERAGE RESULTS")
    print("="*80)
    print(f"Average SacreBLEU: {(nep2hin_results['sacrebleu'] + hin2nep_results['sacrebleu']) / 2:.4f}")
    print(f"Average ChrF:      {(nep2hin_results['chrf'] + hin2nep_results['chrf']) / 2:.4f}")
    print(f"Average ChrF++:    {(nep2hin_results['chrfpp'] + hin2nep_results['chrfpp']) / 2:.4f}")
    print(f"Average METEOR:    {(nep2hin_results['meteor'] + hin2nep_results['meteor']) / 2:.4f}")
    print(f"Average COMET:     {(nep2hin_results['comet'] + hin2nep_results['comet']) / 2:.4f}")
    print("="*80 + "\n")

if __name__ == "__main__":
    main()